# EEI6366 Lab 01: Big Data Processing with PySpark

**Dataset:** `open_university_student_performance.csv`  
**Size:** 150,500 rows  

This notebook uses **simple and easy operations**.

- Load and clean the dataset
- Explore using **PySpark**: 4 easy operations
- Explore using **Spark SQL**: 4 easy operations

## 1. Install and Start PySpark

In [ ]:
!pip install -q pyspark findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Lab01-Student-Performance").getOrCreate()
spark

## 2. Load the Dataset

In [ ]:
from google.colab import files
print("Upload open_university_student_performance.csv")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = spark.read.csv(file_name, header=True, inferSchema=True)

print("Dataset loaded successfully")
print("Total rows:", df.count())
df.show(5)

## 3. Explore Dataset Structure

In [ ]:
# Show column names and data types
df.printSchema()

# Show number of columns
print("Number of columns:", len(df.columns))
print("Column names:", df.columns)

## 4. Check Missing Values

In [ ]:
# Count null values in each column
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

## 5. Check and Remove Duplicates

In [ ]:
# Check how many duplicate rows exist
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", total_rows - unique_rows)

# Remove duplicates
df = df.dropDuplicates()
print("After removing duplicates:", df.count())

## 6. Handle Missing Values

In [ ]:
# Drop rows that have any missing values
df = df.dropna()

print("Rows after dropping missing values:", df.count())

## 7. Verify Cleaned Dataset

In [ ]:
print("Final number of rows:", df.count())
print("Final number of columns:", len(df.columns))

# Check again for missing values (should be 0)
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

df.show(5)

## 8. PySpark Operations (4 Easy Operations)

### Operation 1 – Show Schema and Count

See the structure of the cleaned data and total number of rows.

In [ ]:
df.printSchema()
print("Total rows:", df.count())

### Operation 2 – Filter Data

Show only the students who **Failed**.

In [ ]:
failed_students = df.filter(df.status == "Fail")
failed_students.show(10)
print("Number of Fail students:", failed_students.count())

### Operation 3 – Select Columns and Order

Select only a few columns and sort by overall mark (highest first).

In [ ]:
df.select("student_id", "course_code", "overall_mark", "grade") \
  .orderBy(F.desc("overall_mark")) \
  .show(10)

### Operation 4 – GroupBy and Average

Find the average overall mark for each center.

In [ ]:
df.groupBy("center") \
  .avg("overall_mark") \
  .show()

## 9. Spark SQL Operations (4 Easy Operations)

First create a temporary view so we can use SQL.

In [ ]:
df.createOrReplaceTempView("students")

### SQL Operation 1 – Simple SELECT with WHERE

Show students from Colombo center who got grade A or A+.

In [ ]:
spark.sql("""
SELECT student_id, course_code, overall_mark, grade
FROM students
WHERE center = 'Colombo' AND grade IN ('A', 'A+')
LIMIT 10
""").show()

### SQL Operation 2 – COUNT by Status

Count how many students Passed and how many Failed.

In [ ]:
spark.sql("""
SELECT status, COUNT(*) AS total
FROM students
GROUP BY status
""").show()

### SQL Operation 3 – Average Mark by Course

Find the average overall mark for each course.

In [ ]:
spark.sql("""
SELECT course_code, ROUND(AVG(overall_mark), 2) AS avg_mark
FROM students
GROUP BY course_code
ORDER BY avg_mark DESC
""").show()

### SQL Operation 4 – Highest Mark in Each Center

Find the maximum overall mark for every center.

In [ ]:
spark.sql("""
SELECT center, MAX(overall_mark) AS highest_mark
FROM students
GROUP BY center
ORDER BY highest_mark DESC
""").show()

## 10. Findings and Challenges

### Findings
- The dataset has about **150,500** student records.
- There are 6 courses and 6 regional centers.
- Most students have **Pass** status.
- Average marks are different across centers and courses.
- Some students got very high marks (A / A+).

### Challenges
- Setting up PySpark in Google Colab for the first time.
- Understanding how to check and remove missing values and duplicates.
- Learning the difference between PySpark DataFrame methods and Spark SQL.
- Choosing simple operations that are easy to explain in the viva.

In [ ]:
spark.stop()
print("Spark stopped.")